# Урок 3.2. RAG в продакшене: Qdrant, асинхронность и FastAPI

**Модуль 3 «RAG: Retrieval-Augmented Generation на практике» · Курс «LLM для Python-разработчиков»**

В уроке 3.1 RAG жил в ноутбуке: индекс пересобирался при каждом запуске, пользователь был один — вы, а все вызовы блокировали друг друга (и это никого не смущало). Сегодня превращаем прототип в **сервис**: индекс переезжает в Qdrant, вызовы становятся асинхронными, а наружу появляется HTTP-API. Код сервиса лежит в `service/` — в уроке мы разберём его по частям и погоняем живьём.

## 🎯 Цели урока

После этого урока вы сможете:

- объяснить, чем **сервис** отличается от прототипа — по каждому из пяти пунктов;
- спроектировать **эмбеддер как интерфейс** и понять, почему у TF-IDF есть состояние, а у e5 — нет;
- вынести индексацию в **отдельный батч-скрипт** и объяснить, почему ей не место в старте приложения;
- использовать **async/await** там, где это даёт пользу, — и показать измерением, где не даёт;
- собрать **FastAPI-сервис**: lifespan, Pydantic-валидация, healthz, стриминг, защита GPU семафором;
- воспроизвести главную аварию async-мира — **блокировку event loop** — и увидеть её в миллисекундах.

## План

1. Чем сервис отличается от прототипа
2. Эмбеддер как интерфейс: TF-IDF и e5 за одним контрактом
3. Индексация — батч, а не старт приложения
4. Асинхронность: официант и кухня
5. FastAPI-сервис: разбор `service/app.py`
6. Ловушка: синхронный вызов внутри `async def`
7. Стриминг по HTTP: Server-Sent Events

⏱ **Время:** ~90–120 минут.

> 💡 Понадобятся: контейнер Qdrant (`docker compose up -d` из папки модуля — подойдёт и контейнер модуля 2, порты общие), Ollama с `qwen2.5:3b` и пакеты `fastapi`, `uvicorn`, `qdrant-client` (все есть в `requirements.txt`). Семантический эмбеддер e5 по-прежнему опционален: сервис честно работает на TF-IDF, а апгрейд — одна переменная окружения.

## 1. Чем сервис отличается от прототипа

Ноутбук 3.1 отвечал на вопросы не хуже, чем будет отвечать сервис. Разница не в качестве ответов, а в **условиях эксплуатации**:

| | прототип (ноутбук 3.1) | сервис (этот урок) |
|---|---|---|
| индекс | пересобирается при каждом запуске, живёт в RAM | лежит в Qdrant, переживает рестарты |
| пользователи | один — вы | сколько придёт, и все одновременно |
| вызовы Qdrant и LLM | блокирующие (и пусть) | асинхронные: ожидание не останавливает сервис |
| падение зависимости | стектрейс в лицо | код 503 и честный `/healthz` |
| конфигурация | константы в ячейке | переменные окружения |

Архитектура делится на два независимых конвейера — они даже запускаются в разное время:

```text
        ОНЛАЙН (сервис, живёт всегда)                ОФЛАЙН (батч, по команде)

┌────────┐  POST /ask /search /healthz  ┌─────────────┐      ┌─────────────┐
│ клиент │ ───────────────────────────> │   FastAPI   │      │ index_kb.py │
│  (кто  │ <─────────────────────────── │ service/app │      └──────┬──────┘
│угодно) │        JSON или SSE          └──┬───────┬──┘             │ чанкинг +
└────────┘                                 │       │                │ fit эмбеддера +
                             await поиск   │       │ await          │ upsert векторов
                                           v       │ генерация      v
                                    ┌────────────┐ │         (та же Qdrant)
                                    │   Qdrant   │ │
                                    └────────────┘ v
                                             ┌────────────┐
                                             │   Ollama   │
                                             └────────────┘
```

Код сервиса — три файла в `service/`:

```text
service/
├── embedder.py   # эмбеддеры за одним интерфейсом: TfidfEmbedder, E5Embedder
├── index_kb.py   # батч-индексация: данные -> чанки -> векторы -> Qdrant
└── app.py        # FastAPI: /healthz, /search, /ask, /ask/stream (+одна ловушка)
```

Механика RAG внутри — **дословно из урока 3.1**: `retrieve -> build_prompt -> chat`, тот же системный промпт, те же grounding-правила. Сервисная обвязка не меняет сути пайплайна — и это важный урок сам по себе.

In [ ]:
import json
import os
import subprocess
import sys
import time

import httpx

sys.path.insert(0, os.getcwd())     # чтобы import service.* работал и вне Jupyter

# везде 127.0.0.1, а НЕ localhost - это не прихоть, а +2 секунды на каждое
# соединение (реальная история этого урока - см. таблицу частых ошибок)
QDRANT_URL = os.getenv("QDRANT_URL", "http://127.0.0.1:6333")
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://127.0.0.1:11434")
MODEL = os.getenv("LLM_MODEL", "qwen2.5:3b")
SERVICE_URL = "http://127.0.0.1:8010"           # наш сервис поднимем чуть позже
COLLECTION = "vectorika_rag_e5"


def check(label: str, probe) -> bool:
    try:
        detail = probe()
        print(f"[ OK ] {label}: {detail}")
        return True
    except Exception as error:
        print(f"[SKIP] {label}: {type(error).__name__} - зависимые ячейки пропустятся")
        return False


def _deps():
    import fastapi          # noqa: F401
    import qdrant_client    # noqa: F401
    import uvicorn          # noqa: F401
    return "установлены"


def _qdrant():
    return "v" + httpx.get(QDRANT_URL, timeout=3).json()["version"]


def _ollama():
    models = [m["name"] for m in
              httpx.get(f"{OLLAMA_URL}/api/tags", timeout=3).json()["models"]]
    assert MODEL in models, f"модель {MODEL} не скачана"
    return MODEL


SERVICE_DEPS = check("пакеты fastapi + uvicorn + qdrant-client", _deps)
QDRANT_AVAILABLE = check("Qdrant (docker compose up -d)", _qdrant)
OLLAMA_AVAILABLE = check("Ollama и модель (урок 1.5)", _ollama)

## 2. Эмбеддер как интерфейс

Сервису всё равно, кто превращает текст в вектор. Ему важен **контракт** — и в `service/embedder.py` он описан прямо в докстринге:

```text
emb.dim                      -> размерность вектора
emb.fit(texts)               -> обучиться на корпусе (нейросетевым - no-op)
emb.embed_docs(texts)        -> векторы документов
emb.embed_query(text)        -> вектор запроса
emb.save(path) / .load(path) -> состояние на диск и обратно
```

Под контрактом живут две реализации, и их сравнение — почти вся теория этого раздела:

| | `TfidfEmbedder` | `E5Embedder` |
|---|---|---|
| откуда знания | **из нашего корпуса**: словарь + IDF считаются в `fit()` | выучены заранее на чужих терабайтах, `fit()` — заглушка |
| состояние | словарь и IDF надо **сохранять на диск** и версионировать вместе с коллекцией | состояние = веса модели в кэше HF |
| вектор | 902 координаты (по числу слов), почти все нули | 384 плотных координаты |
| скорость кодирования | микросекунды | десятки миллисекунд на CPU |
| синонимы | не понимает («Упал прод» — мимо, модуль 2 это доказал) | понимает |
| префиксы | не нужны | `query:` / `passage:` обязательны (урок 2.1) |

Почему это работает: TF-IDF-вектор, L2-нормированный, — это классическая векторная модель поиска, и **косинус между такими векторами Qdrant считает точно так же, как между e5-эмбеддингами**. Базе всё равно, откуда числа. А вот смешивать их нельзя: правило «одна коллекция — одна модель» из урока 2.1 здесь превращается в суффикс имени — `vectorika_rag_tfidf` и `vectorika_rag_e5` живут раздельно. Запрос, закодированный не тем эмбеддером, не упадёт с ошибкой — он просто молча вернёт мусор. Это худший вид бага.

Посмотрим на e5-вектор запроса вблизи:

In [ ]:
from service.embedder import E5Embedder
from service.index_kb import build_chunks

chunks = build_chunks()      # те же 31 чанк с контекстными заголовками, что в 3.1
embedder = E5Embedder()
print(f"чанков: {len(chunks)}, слов в словаре: {embedder.dim} "
      f"-> вектор из {embedder.dim} координат")

In [ ]:
vector = embedder.embed_query("Сколько дней отпуска положено в году?")

## 3. Индексация — батч, а не старт приложения

Соблазн знакомый: «положу индексацию в старт сервиса, и всё всегда свежее». В `service/index_kb.py` она сознательно вынесена в отдельный скрипт, и на то три причины:

1. **Индекс живёт дольше процесса.** Сервис рестартует при деплое, падении, обновлении — база знаний при этом не менялась. Переиндексация на каждый рестарт — это минуты недоступности на ровном месте.
2. **Переиндексация — это решение, а не побочный эффект.** Её запускает человек или планировщик, когда обновились документы. У неё свой ритм жизни.
3. **Реплики.** Когда сервисов станет два, индексация при старте превратится в гонку: оба пересоздают коллекцию, один затирает работу другого.

Внутри скрипта — знакомый конвейер: `load_documents -> chunk_paragraphs` (уроки 2.5–2.6, те же чанки с контекстными заголовками, что в 3.1), затем `fit` эмбеддера, `create_collection` с **cosine** и `upsert` с **uuid5-идентификаторами** (Qdrant принимает только int/UUID — урок 2.4; детерминированный uuid5 из `chunk_id` делает повторный запуск обновлением, а не дублированием).

И одна деталь, которую легко упустить: скрипт **сохраняет состояние TF-IDF-эмбеддера** в `service/tfidf_state.json`. Запрос обязан кодироваться тем же словарём, что и документы, — иначе оси векторов «поедут» и косинус потеряет смысл. Состояние эмбеддера — часть индекса, версионируются они вместе.

Заметьте: клиент Qdrant в скрипте — **синхронный**. Батчу некого обслуживать, пока он ждёт базу, — асинхронность ему ничего не даёт. Это первый ответ на вопрос «где async нужен»: не везде.

In [ ]:
if QDRANT_AVAILABLE:
    result = subprocess.run(
        [sys.executable, "service/index_kb.py", "--backend", "e5"],
        capture_output=True, text=True, encoding="utf-8",
    )
    print(result.stdout.strip() or result.stderr.strip())
else:
    print("[пропущено: Qdrant недоступен]")

In [ ]:
# --- проверка качества: словарь с диска + 7 вопросов модуля 2 ---
EVAL_QUESTIONS = [
    ("Сколько дней отпуска положено в году?",                 "otpusk"),
    ("Как оформить больничный лист?",                         "bolnichny"),
    ("VPN подключён, но внутренние сайты не открываются",     "vpn"),
    ("Кто должен посмотреть мой код перед мержем?",           "code-review"),
    ("Упал прод, что делать в первую очередь?",               "incidents"),
    ("Сколько компания доплачивает в первые дни болезни?",    "bolnichny"),
    ("За сколько минут дежурный должен отреагировать ночью?", "incidents"),
]

if QDRANT_AVAILABLE:
    from qdrant_client import QdrantClient

    sync_client = QdrantClient(url=QDRANT_URL)
    # emb_from_disk = TfidfEmbedder.load("service/tfidf_state.json")  # состояние с диска!

    top1 = top3 = 0
    for question, expected in EVAL_QUESTIONS:
        hits = sync_client.query_points(
            COLLECTION, query=embedder.embed_query(question), limit=3,
        ).points
        got = [h.payload["doc_id"] for h in hits]
        top1 += got[0] == expected
        top3 += expected in got
        mark = " да " if got[0] == expected else ("топ3" if expected in got else "МИМО")
        print(f"[{mark}] {question[:48]:<50} -> {got[0]} ({hits[0].score:.3f})")
    print(f"\ntop-1: {top1}/7, top-3: {top3}/7")
else:
    print("[пропущено: Qdrant недоступен]")

У нас вышло **top-1 6/7** — ровно столько же, сколько у лексического поиска в модуле 2, и промахивается тот же вопрос: «Упал прод» (синонимы, TF-IDF их не знает). Это правильный результат: **векторная база не добавляет и не убавляет качества поиска** — она добавляет персистентность, конкурентный доступ и фильтры. Качество определяет эмбеддер; станет он e5 — и «Упал прод» найдётся, база при этом не изменится ни на строчку.

## 4. Асинхронность: официант и кухня

Представьте официанта, который принял заказ, отнёс его на кухню — и **стоит у плиты, пока готовится стейк**. Зал ждёт, заказы копятся, а он смотрит на сковородку. Это синхронный обработчик: пока `httpx.post` ждёт ответа Ollama (секунды!), поток занят ничем.

Асинхронный официант отдаёт заказ на кухню и возвращается в зал. `await` — это ровно «отдал и вернулся»: функция говорит event loop'у «я жду внешнюю систему, займись другими». Один поток, один цикл событий — и десятки обслуживаемых клиентов, потому что 99% времени RAG-запроса — это ожидание Qdrant и Ollama, а не работа CPU.

Два честных предупреждения, прежде чем писать код:

- **Async не ускоряет единичный вызов.** Один `await client.query_points(...)` занимает столько же, сколько синхронный аналог. Выгода появляется, только когда ожиданий много и они перекрываются.
- **Async не помогает CPU-работе.** Если обработчик считает что-то тяжёлое (например, гоняет e5 на CPU), event loop стоит так же, как при блокирующем вызове. Для такого есть `anyio.to_thread.run_sync` — унести работу в поток.

В нашем сервисе асинхронными станут два ожидания: поиск (`AsyncQdrantClient` — тот же API, что у синхронного клиента из урока 2.4, только с `await`) и генерация (`httpx.AsyncClient`). Проверим первое — и заодно убедимся, что чуда со скоростью нет:

In [ ]:
import asyncio
import concurrent.futures


def run_async(coro):
    """asyncio.run, работающий и в скрипте, и в Jupyter (урок 1.4).

    В Jupyter event loop уже крутится, поэтому запускаем корутину
    в отдельном потоке со своим собственным циклом."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coro).result()


if QDRANT_AVAILABLE:
    from qdrant_client import AsyncQdrantClient

    QUESTION = "Как оформить больничный лист?"

    async def async_search_once():
        # клиент создаётся ВНУТРИ корутины: он привязывается к текущему loop
        client = AsyncQdrantClient(url=QDRANT_URL)
        # непомеренный запрос-прогрев: TCP-соединение устанавливается один раз,
        # иначе сравнение с прогретым sync-клиентом будет нечестным
        await client.query_points(COLLECTION,
                                  query=embedder.embed_query(QUESTION), limit=1)
        t0 = time.perf_counter()
        hits = await client.query_points(
            COLLECTION, query=embedder.embed_query(QUESTION), limit=3,
        )
        elapsed_ms = (time.perf_counter() - t0) * 1000
        await client.close()
        return hits.points[0].payload["doc_id"], elapsed_ms

    found_doc, ms_async = run_async(async_search_once())

    t0 = time.perf_counter()
    sync_client.query_points(COLLECTION, query=embedder.embed_query(QUESTION), limit=3)
    ms_sync = (time.perf_counter() - t0) * 1000

    print(f"sync поиск:  {ms_sync:5.1f} мс")
    print(f"async поиск: {ms_async:5.1f} мс -> нашёл {found_doc}")
    print("\nБыстрее не стало - и не должно было. Выгода в другом:")
    print("пока await ждёт Qdrant или Ollama, сервис обслуживает остальных.")
    print("Насколько это важно - увидим на живом сервисе через пару ячеек.")
else:
    print("[пропущено: Qdrant недоступен]")

## 5. FastAPI-сервис: разбор `service/app.py`

Файл длинный, но состоит из четырёх идей. Открывайте его рядом и следите по коду.

**Идея 1: тяжёлые объекты создаются один раз — в lifespan.** Клиент с пулом соединений — дорогая вещь; создавать его на каждый запрос — классическая ошибка (латентность + утечка соединений). FastAPI даёт точку «при старте / при остановке»:

```python
@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.embedder = make_embedder(EMB_BACKEND, state_path)  # словарь с диска
    app.state.qdrant = AsyncQdrantClient(url=QDRANT_URL)
    app.state.ollama = httpx.AsyncClient(base_url=OLLAMA_URL,
                                         timeout=httpx.Timeout(120, connect=5))
    app.state.gen_semaphore = asyncio.Semaphore(1)   # об этом - ниже
    yield                                            # <- здесь сервис живёт
    await app.state.qdrant.close()
    await app.state.ollama.aclose()
```

Обратите внимание на `httpx.Timeout(120, connect=5)`: генерация имеет право быть долгой (120с), но «не могу подключиться» должно выясняться за 5 секунд, а не висеть вечно.

**Идея 2: границу сервиса охраняет Pydantic.** Модель запроса — это валидация, документация и сериализация в одном флаконе (знакомо по уроку 1.8, где Pydantic проверял ответы LLM):

```python
class AskRequest(BaseModel):
    question: str = Field(min_length=3, max_length=500)
    top_k: int = Field(default=3, ge=1, le=10)
    temperature: float = Field(default=0.2, ge=0.0, le=1.0)
```

Вопрос длиннее 500 символов? Клиент получит 422 с объяснением — до нашего кода запрос даже не дойдёт. Ограничение `top_k <= 10` — это ещё и защита кошелька: помните из решения 3.1, как top_k=6 раздул промпт до 881 токена.

**Идея 3: healthz проверяет зависимости, и проверяет их параллельно.** Сервис, который отвечает «ok», когда Ollama мертва, обманывает балансировщик. Наш `/healthz` опрашивает Qdrant и Ollama через `asyncio.gather` — первое применение конкурентности в деле — и честно отвечает `degraded`, если что-то отвалилось. Когда зависимость оживёт, статус сам вернётся в `ok` — рестарт сервиса не нужен.

**Идея 4: GPU одна — генерации выстраиваются в очередь.** Если пять клиентов одновременно позовут `/ask`, пять параллельных генераций устроят давку за VRAM (а Ollama по умолчанию и так их сериализует, но молча). Мы делаем очередь **явной и измеримой**:

```python
async with request.app.state.gen_semaphore:      # ждём своей очереди к GPU
    t_gen = time.perf_counter()                  # очередь кончилась
    answer = await ollama_chat(request, messages, body.temperature)
```

а в ответе клиенту — разбивка `search_ms / queue_ms / llm_ms`: сколько занял поиск, сколько простояли в очереди, сколько генерировала модель. Латентность, разложенная по полочкам, — первый шаг к наблюдаемости (вся глубина — в уроке 3.7).

Запускаем. Ячейка умеет два сценария: если сервис уже запущен у вас в терминале (`uvicorn service.app:app --port 8010`) — переиспользует его; иначе поднимет свой процесс:

In [ ]:
service_proc = None
SERVICE_UP = False


def service_alive() -> bool:
    try:
        return httpx.get(f"{SERVICE_URL}/healthz", timeout=2).status_code == 200
    except httpx.HTTPError:
        return False


if service_alive():
    SERVICE_UP = True
    print("сервис уже запущен - переиспользуем его")
elif SERVICE_DEPS and QDRANT_AVAILABLE:
    service_proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "service.app:app",
         "--host", "127.0.0.1", "--port", "8010", "--log-level", "warning"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    for _ in range(40):                     # до ~20 секунд на старт
        if service_alive():
            SERVICE_UP = True
            break
        time.sleep(0.5)
    print("сервис поднят на " + SERVICE_URL if SERVICE_UP
          else "сервис не поднялся: запустите uvicorn в терминале и посмотрите лог")
else:
    print("[пропущено: не хватает зависимостей или Qdrant]")

if SERVICE_UP and OLLAMA_AVAILABLE:
    t0 = time.perf_counter()
    httpx.post(f"{SERVICE_URL}/ask", json={"question": "Привет!", "top_k": 1}, timeout=300)
    print(f"прогрев модели: {time.perf_counter() - t0:.1f}с "
          f"(холодный старт - модель грузится в память, урок 1.5)")

In [ ]:
if SERVICE_UP:
    health = httpx.get(f"{SERVICE_URL}/healthz", timeout=10).json()
    print("GET /healthz:")
    print(json.dumps(health, ensure_ascii=False, indent=2))

    found = httpx.post(
        f"{SERVICE_URL}/search",
        json={"question": "Кто должен посмотреть мой код перед мержем?"},
        timeout=30,
    ).json()
    print("\nPOST /search:")
    for source in found:
        print(f"  [{source['score']:.3f}] {source['chunk_id']:<16} {source['text'][:58]}...")
else:
    print("[пропущено: сервис не запущен]")

In [ ]:
if SERVICE_UP and OLLAMA_AVAILABLE:
    data = httpx.post(
        f"{SERVICE_URL}/ask",
        json={"question": "Сколько компания доплачивает в первые дни болезни?"},
        timeout=300,
    ).json()
    print("ответ:", data["answer"])
    print("источники:", [s["chunk_id"] for s in data["sources"]])
    total_ms = data["search_ms"] + data["queue_ms"] + data["llm_ms"]
    print(f"\nлатентность: поиск {data['search_ms']} мс "
          f"+ очередь {data['queue_ms']} мс + LLM {data['llm_ms']} мс")
    print(f"на поиск пришлось {data['search_ms'] / total_ms:.1%} времени: "
          f"цена RAG-ответа - это генерация, а не поиск")
else:
    print("[пропущено: нужен сервис и Ollama]")

## 6. Ловушка: синхронный вызов внутри `async def`

Самая частая и самая коварная ошибка async-кода. Обработчик объявлен `async def`, FastAPI доверчиво запускает его в event loop — а внутри кто-то написал синхронный `httpx.post` (или `requests.get`, или тяжёлый `model.encode`). Официант **уснул у плиты**: пока вызов не вернётся, event loop не делает НИЧЕГО. Все клиенты — включая безобидный `/healthz` — молча ждут.

Коварство в том, что ошибка **не видна без нагрузки**: один клиент в вакууме получает свой ответ как ни в чём не бывало. В `service/app.py` специально оставлен эндпоинт-антипример `/ask-blocking` — он отличается от `/ask` одной строкой:

```python
response = httpx.post(f"{OLLAMA_URL}/api/chat", json={...}, timeout=120)  # БЛОКИРУЕТ LOOP
```

Устроим обоим эндпоинтам одинаковую проверку: отправляем вопрос, а параллельно каждые 0.4 секунды дёргаем `/healthz` и меряем, за сколько он отвечает.

In [ ]:
if SERVICE_UP and OLLAMA_AVAILABLE:
    # вопрос, вынуждающий ДЛИННЫЙ ответ: пока модель пишет, пробы успеют накопиться
    PROBE_QUESTION = ("Объясни подробно, по пунктам: как оформить отпуск, "
                      "больничный и удалённую работу?")

    async def probes_during(endpoint: str) -> list[float]:
        """POST на endpoint + восемь health-проб с шагом 0.35с, всё одновременно."""
        async with httpx.AsyncClient(base_url=SERVICE_URL, timeout=300) as client:
            await client.get("/healthz")            # прогрев соединения

            async def probe(i: int) -> float:
                await asyncio.sleep(0.35 * i)
                t0 = time.perf_counter()
                await client.get("/healthz", timeout=120)
                return (time.perf_counter() - t0) * 1000

            results = await asyncio.gather(
                client.post(endpoint, json={"question": PROBE_QUESTION, "top_k": 5}),
                *[probe(i) for i in range(8)],
            )
            return results[1:]

    for endpoint in ("/ask", "/ask-blocking"):
        latencies = run_async(probes_during(endpoint))
        print(f"{endpoint:<14} healthz, мс: "
              + " ".join(f"{v:>5.0f}" for v in latencies))
else:
    print("[пропущено: нужен сервис и Ollama]")

В нашем прогоне получилось так:

```text
/ask           healthz, мс:    23    20    15    13    14    13    16    14
/ask-blocking  healthz, мс:  5637  5336  4990  4643  4297  3949  3601  3255
```

Пока `/ask` генерировал длинный ответ, `/healthz` отвечал за 13–23 миллисекунды — event loop свободен, `await` отпустил его на время генерации. А у `/ask-blocking` — **лесенка**: все восемь проб зависли и отпустились одним махом, когда синхронный вызов вернулся. Числа убывают ровно на шаг запуска (0.35с): кто позже встал в очередь, тот меньше прождал. Первая проба ждала **всю генерацию целиком** — 5.6 секунды. Снаружи это выглядит как «сервис завис»: ровно так его увидят мониторинг, балансировщик и живые пользователи.

Что делать, если внутри обработчика всё-таки есть тяжёлая **CPU-работа** (например, `e5.encode` на процессоре)? `await` тут не поможет — ждать нечего, надо считать. Рецепт: унести в поток — `await anyio.to_thread.run_sync(embedder.embed_query, question)`. Event loop свободен, считает соседний поток. Для нашего TF-IDF это не нужно: его `embed_query` занимает микросекунды (мы измерили: ~0.1 мс), дешевле посчитать на месте, чем платить за переключение потока.

## 7. Стриминг: Server-Sent Events

В уроке 3.1 стриминг спасал ощущение скорости: первые токены — через полсекунды, а не «весь ответ через десять». Сервису нужно то же самое, но **через HTTP-границу**. Стандартный способ — SSE (Server-Sent Events): обычный HTTP-ответ с `Content-Type: text/event-stream`, в котором события идут строками `data: {...}` по мере готовности. Именно так стримят ChatGPT и OpenAI-совместимые API (урок 1.3).

Наш `/ask/stream` устроен как конвейер с двумя приятными деталями:

1. **Источники уходят первым событием** — ещё до первого токена. Клиент может показать «нашёл в: больничные, отпуск...» мгновенно, пока модель думает.
2. Токены пересылаются по мере прихода из Ollama: `async for line in response.aiter_lines()` читает стрим Ollama, а `yield` немедленно отдаёт событие клиенту. Сервис здесь — насос, а не буфер.

```python
async def event_stream():
    yield f"data: {json.dumps({'sources': ...})}\n\n"        # источники сразу
    async with ... .stream("POST", "/api/chat", ...) as r:    # стрим из Ollama
        async for line in r.aiter_lines():
            yield f"data: {json.dumps({'delta': ...})}\n\n"  # токен - клиенту

return StreamingResponse(event_stream(), media_type="text/event-stream")
```

In [ ]:
if SERVICE_UP and OLLAMA_AVAILABLE:
    t0 = time.perf_counter()
    first_token_at = None
    with httpx.stream(
        "POST", f"{SERVICE_URL}/ask/stream",
        json={"question": "Сколько дней отпуска положено в году?"},
        timeout=300,
    ) as response:
        for line in response.iter_lines():
            if not line.startswith("data: "):
                continue
            event = json.loads(line[len("data: "):])
            if "sources" in event:
                at_ms = (time.perf_counter() - t0) * 1000
                names = [s["chunk_id"] for s in event["sources"]]
                print(f"[{at_ms:.0f} мс] источники: {names}")
                print("ответ: ", end="")
            elif "delta" in event:
                if first_token_at is None:
                    first_token_at = time.perf_counter() - t0
                print(event["delta"], end="", flush=True)
            elif event.get("done"):
                print()
    print(f"\nисточники пришли ДО генерации, первый токен - через {first_token_at:.2f}с, "
          f"весь ответ - за {time.perf_counter() - t0:.2f}с")
else:
    print("[пропущено: нужен сервис и Ollama]")

## ⚠️ Частые ошибки

| ошибка | чем оборачивается | лекарство |
|---|---|---|
| создавать `AsyncQdrantClient` / `httpx.AsyncClient` на каждый запрос | +латентность на рукопожатия, утечка соединений | создать в `lifespan`, хранить в `app.state` |
| синхронный вызов внутри `async def` | event loop стоит, ВСЕ клиенты ждут (лесенка из раздела 6) | async-клиенты и `await`; для CPU-работы — `anyio.to_thread.run_sync` |
| индексация при старте сервиса | каждый рестарт — минуты недоступности; реплики затирают друг друга | отдельный батч-скрипт (`index_kb.py`) |
| потерять состояние TF-IDF (или закодировать запрос другим эмбеддером) | выдача превращается в мусор **без единой ошибки** | состояние эмбеддера версионируется вместе с коллекцией; суффикс бэкенда в имени |
| httpx без таймаутов | зависшая Ollama держит запросы вечно | `httpx.Timeout(120, connect=5)` |
| ходить на `localhost`, когда сервер слушает только IPv4 | у `localhost` **два** адреса: httpx сперва пробует IPv6 `::1`, получает отказ и лишь потом идёт на `127.0.0.1` — на Windows это ~2с штрафа на **каждое** новое соединение. Мы поймали это хронометражем SSE: «мгновенные» источники приходили через 2.2с, и виноват оказался не сервис, а connect | писать `127.0.0.1` явно; держать keep-alive клиент — пул из `lifespan` платит за connect один раз |
| `/healthz`, который всегда «ok» | балансировщик шлёт трафик в мёртвый сервис | проверять зависимости (у нас — `gather` по Qdrant и Ollama) |
| пускать генерации на GPU без очереди | давка за VRAM, непредсказуемая латентность | семафор + честный `queue_ms` в ответе |
| «в Jupyter работало, в скрипте упало» с `asyncio.run` | в Jupyter уже крутится event loop | хелпер `run_async` (урок 1.4) или `await` прямо в ячейке |

## 🏋️ Упражнения

**1. Eval через HTTP.** У сервиса появилась граница — значит, качество можно мерить снаружи, как это сделает любой клиент. Прогоните 7 вопросов `EVAL_QUESTIONS` через `POST /search` и посчитайте top-1/top-3 по `doc_id`. Совпадёт ли с проверкой из раздела 3 (обязано!)?

**2. Очередь глазами клиентов.** Отправьте **три** `/ask` одновременно (`asyncio.gather`). Предскажите до запуска: какие `queue_ms` увидит каждый клиент при семафоре в 1? Проверьте — и заодно убедитесь, что `/healthz` жив, пока все трое стоят в очереди.

**3. Переезд на e5 (бумажное, если модель не скачана).** Перечислите все шаги перевода сервиса с TF-IDF на e5 — от скачивания модели до перезапуска. Какая строка в `retrieve()` из `app.py` станет проблемой под нагрузкой и как её лечить? Напишите патч.

## 📝 Мини-квиз

<details><summary><b>1. Почему индексацию не кладут в lifespan сервиса?</b></summary>

Индекс живёт дольше процесса: сервис рестартует десятки раз, а документы не менялись. Индексация при старте — это недоступность на каждом рестарте, а при нескольких репликах — ещё и гонка (обе пересоздают коллекцию). Переиндексация — отдельное решение с отдельным запуском: батч-скрипт, руками или по расписанию.
</details>

<details><summary><b>2. Обработчик объявлен async def, но внутри — requests.get(...). Что почувствуют остальные клиенты сервиса?</b></summary>

Всё встанет: event loop один, и пока синхронный вызов не вернётся, loop не обрабатывает никого — даже `/healthz`. Снаружи это выглядит как «сервис завис» ровно на время вызова (лесенка из раздела 6). Причём без нагрузки ошибка невидима — одинокий клиент получает свой ответ нормально.
</details>

<details><summary><b>3. Зачем TF-IDF-эмбеддеру файл состояния, а e5 обходится без него?</b></summary>

Словарь и IDF выучиваются из **нашего** корпуса при индексации. Запрос обязан кодироваться тем же словарём — иначе оси не совпадут и косинус потеряет смысл. У e5 «состояние» — это веса модели, выученные заранее и лежащие в кэше HF: наш корпус на кодирование не влияет.
</details>

<details><summary><b>4. /healthz вернул {"status": "degraded", "points": 31, "model": null}. Что случилось и надо ли рестартовать сервис?</b></summary>

Qdrant жив (точки на месте), а Ollama не отвечает — умерла, перезапускается или обновляется. Рестартовать сервис не нужно: healthz проверяет зависимости на каждый вызов, и когда Ollama вернётся, статус сам станет `ok`. Рестарт ничего не ускорит — сервис не сломан, сломана зависимость. (Реальный случай: во время подготовки этого урока Ollama самообновилась посреди работы, и healthz честно отыграл `degraded -> ok` без единого рестарта.)
</details>

<details><summary><b>5. Зачем семафор перед вызовом Ollama, если она и так умеет очередь?</b></summary>

Затем, чтобы очередь была **нашей и измеримой**. Семафор ограничивает конкуренцию за GPU на нашей стороне, а разница `queue_ms` в ответе показывает клиенту (и мониторингу!), сколько он простоял. Очередь внутри Ollama невидима — с семафором «почему медленно?» перестаёт быть загадкой: смотрим, очередь это или генерация.
</details>

<details><summary><b>6. Почему коллекция называется vectorika_rag_tfidf, а не просто vectorika_rag?</b></summary>

«Одна коллекция — одна модель» (урок 2.1). Вектор запроса от другого эмбеддера не упадёт с ошибкой (если размерности совпали) — он молча вернёт мусор. Суффикс бэкенда в имени делает несовместимость видимой: e5-сервис просто не найдёт «своей» коллекции и честно отдаст 503, а не тихую чушь.
</details>

## Решения

**Решение 1 — eval через HTTP.** Обратите внимание: числа обязаны совпасть с разделом 3 — это тот же индекс и тот же эмбеддер, просто между нами и Qdrant теперь стоит сервис.

In [ ]:
if SERVICE_UP:
    top1 = top3 = 0
    for question, expected in EVAL_QUESTIONS:
        found = httpx.post(f"{SERVICE_URL}/search",
                           json={"question": question}, timeout=30).json()
        got = [s["doc_id"] for s in found]
        top1 += got[0] == expected
        top3 += expected in got
        mark = " да " if got[0] == expected else ("топ3" if expected in got else "МИМО")
        print(f"[{mark}] {question[:48]:<50} -> {got[0]}")
    print(f"\nчерез HTTP: top-1 {top1}/7, top-3 {top3}/7 - как напрямую из раздела 3")
else:
    print("[пропущено: сервис не запущен]")

**Решение 2 — очередь глазами клиентов.** Предсказание при семафоре в 1: генерации выполняются по очереди, поэтому первый клиент почти не ждёт (`queue_ms ~ 0`), второй ждёт одну генерацию, третий — две. Полное время последнего ≈ сумме трёх генераций. При этом `/healthz` обязан отвечать быстро: очередь стоит к GPU, а не к event loop'у.

In [ ]:
if SERVICE_UP and OLLAMA_AVAILABLE:

    async def three_clients_and_probe():
        async with httpx.AsyncClient(base_url=SERVICE_URL, timeout=600) as client:

            async def one(i: int):
                t0 = time.perf_counter()
                response = await client.post(
                    "/ask", json={"question": "Как оформить больничный лист?"})
                data = response.json()
                return i, time.perf_counter() - t0, data["queue_ms"], data["llm_ms"]

            async def health_probe():
                await asyncio.sleep(1.0)          # в разгар очереди
                t0 = time.perf_counter()
                await client.get("/healthz", timeout=60)
                return (time.perf_counter() - t0) * 1000

            *clients, health_ms = await asyncio.gather(
                one(1), one(2), one(3), health_probe())
            return clients, health_ms

    clients, health_ms = run_async(three_clients_and_probe())
    for i, full_s, queue_ms, llm_ms in sorted(clients):
        print(f"клиент {i}: всего {full_s:5.1f}с = очередь {queue_ms / 1000:4.1f}с "
              f"+ генерация {llm_ms / 1000:4.1f}с (+ поиск)")
    print(f"\n/healthz в разгар очереди: {health_ms:.0f} мс - сервис жив,")
    print("очередь стоит к GPU, а не к event loop'у")
else:
    print("[пропущено: нужен сервис и Ollama]")

**Решение 3 — переезд на e5.** Шаги:

1. Скачать модель (единственный шаг, требующий интернета): `pip install sentence-transformers`, затем любой вызов `E5Embedder` подтянет `intfloat/multilingual-e5-small` (~120 МБ) с Hugging Face в кэш.
2. Переиндексировать в **отдельную** коллекцию: `python service/index_kb.py --backend e5` — создаст `vectorika_rag_e5` с векторами по 384 координаты. Старая TF-IDF-коллекция не пострадает: можно переключаться туда-обратно и сравнивать.
3. Перезапустить сервис с `EMB_BACKEND=e5` — фабрика `make_embedder` подхватит e5, имя коллекции соберётся само. Ни строчки кода менять не нужно — в этом и была цель интерфейса из раздела 2.

Проблемная строка в `retrieve()`:

```python
vector = request.app.state.embedder.embed_query(question)
```

Для TF-IDF это микросекунды, а `e5.encode` на CPU — **десятки миллисекунд чистой CPU-работы прямо в event loop'е**. Под нагрузкой сервис получит мини-версию лесенки из раздела 6 — на каждый запрос. Патч:

```python
import anyio

async def retrieve(request: Request, question: str, top_k: int) -> list[Source]:
    embedder = request.app.state.embedder
    if embedder.name == "e5":                          # CPU-тяжёлый - в поток
        vector = await anyio.to_thread.run_sync(embedder.embed_query, question)
    else:                                              # микросекунды - на месте
        vector = embedder.embed_query(question)
    hits = await request.app.state.qdrant.query_points(
        COLLECTION, query=vector, limit=top_k)
    return [Source(**hit.payload, score=hit.score) for hit in hits.points]
```

И последний штрих — прибираем за собой (если сервис поднимал ноутбук, а не вы в терминале):

In [ ]:
if service_proc is not None:
    service_proc.terminate()
    service_proc.wait(timeout=10)
    print("uvicorn остановлен")
elif SERVICE_UP:
    print("сервис запускали не мы - оставляем работать")
else:
    print("сервис и не запускался")

print("коллекция vectorika_rag_e5 остаётся в Qdrant:")
print("индекс переживает рестарты чего угодно - в этом и был смысл переезда")

## Итоги

Прототип из 3.1 стал сервисом, и по дороге мы собрали все «сервисные» привычки:

- **Эмбеддер — интерфейс.** Контракт из пяти методов; TF-IDF (живой, с состоянием из корпуса) и e5 (guarded, состояние — веса) взаимозаменяемы одной переменной окружения. Qdrant всё равно, откуда числа; «одна коллекция — одна модель» превратилось в суффикс имени.
- **Индексация — батч.** Отдельный синхронный скрипт: индекс живёт дольше процесса, а состояние эмбеддера версионируется вместе с коллекцией.
- **Асинхронность — про ожидание, а не про скорость.** Единичный вызов быстрее не стал (мы измерили), зато event loop свободен, пока LLM думает, — и `/healthz` отвечает за миллисекунды во время генерации. Синхронный вызов в `async def` устраивает лесенку из раздела 6.
- **FastAPI-обвязка:** lifespan для тяжёлых объектов, Pydantic на границе, healthz с параллельной проверкой зависимостей, семафор-очередь к GPU с честным `queue_ms`, SSE-стриминг с источниками первым событием.
- **Качество не изменилось: top-1 6/7, и «Упал прод» всё ещё мимо.** База данных не лечит эмбеддер. Лечит e5 — и теперь его подключение стоит одну переменную окружения.

Откуда что взялось:

| приём урока | родом из |
|---|---|
| чанки с контекстными заголовками, TF-IDF | уроки 2.5–2.6 |
| cosine, uuid5, query_points, «одна коллекция — одна модель» | уроки 2.1, 2.4 |
| RAG-промпт с grounding и источниками | урок 3.1 |
| таймауты, ретраи, стриминг, run_async | уроки 1.3, 1.4 |
| метрики вместо впечатлений (eval через HTTP) | урок 1.7 |

**Что дальше.** У сервиса теперь есть латентность в цифрах — `search_ms / queue_ms / llm_ms`. А вот качество ответов мы по-прежнему оцениваем глазами: «вроде отвечает правильно». В уроке 3.3 появятся метрики качества RAG — faithfulness, answer relevancy, context precision/recall — и golden-датасет, по которому их можно считать регулярно, как тесты.